### 문항 1 금융위 주식시세정보 API 호출과 오류 처리

In [ ]:
# 공공데이터포털에서 발급받은 서비스키로 금융위원회 주식시세정보 API를 호출하시오.

# 대상: https://www.data.go.kr/data/15094808/openapi.do

# 서비스키는 .env로 분리하고, .gitignore에 .env를 추가할 것

# 코드에 키를 하드코딩하지 않고 불러와 사용할 것

# 삼성전자(005930) 최근 5영업일 시세를 조회해 JSON을 dict로 파싱할 것

# 다음 세 가지 오류 상황을 각각 구분해 처리할 것

# 인증키 오류 (SERVICE_KEY_IS_NOT_REGISTERED_ERROR)

# 일일 쿼터 초과 (LIMITED_NUMBER_OF_SERVICE_REQUESTS_EXCEEDS_ERROR)

# 필수 파라미터 누락 (INVALID_REQUEST_PARAMETER_ERROR)

# .env.example 파일을 함께 제출할 것 (키 값은 비울 것)

In [38]:
import os
import time
import json
import hashlib
import pandas as pd
import pymysql
from datetime import datetime
from dotenv import load_dotenv

In [39]:
BASE_URL = 'https://apis.data.go.kr/1160100/service/GetStockSecuritiesInfoService'
FUNCTION1 = '/getStockPriceInfo'

load_dotenv()
KEY = os.getenv('OPEN_API_KEY', 'NO_KEY')

In [41]:
BASE_URL = 'https://apis.data.go.kr/1160100/service/GetStockSecuritiesInfoService'
FUNCTION1 = '/getStockPriceInfo'

load_dotenv()
KEY = os.getenv('OPEN_API_KEY', 'NO_KEY')


class OpenAPIKeyError(Exception):
    """등록되지 않은 서비스키"""
class ExceedsError(Exception):
    """서비스 요청제한횟수 초과 에러"""
class PARAMETERERROR(Exception):
    """잘못된 요청 파라메터 에러"""

def fetch(page=1, size=10, **kwargs):
    res = requests.get(BASE_URL+FUNCTION1, params={
        'serviceKey': KEY,
        'numOfRows' : size,
        'pageNo': page,
        'resultType' : 'json',
        **kwargs,
    } )

    res.raise_for_status()
    data = res.json()
    
    if data.get('OpenAPI_ServiceResponse'):
        msg = data.get('OpenAPI_ServiceResponse')['cmmMsgHeader']['errMsg']
        raise OpenAPIKeyError(msg)
    if data['response']['header']['resultCode'] == '22':
        msg = data.get('OpenAPI_ServiceResponse')['cmmMsgHeader']['errMsg']
        raise ExceedsError(msg)
    if data['response']['header']['resultCode'] == '10':
        msg = data.get('OpenAPI_ServiceResponse')['cmmMsgHeader']['errMsg']
        raise PARAMETERERROR(msg)
    
    return data['response']['body']['items']['item']

def main():
    samsung_stock_api = fetch(page=1, size=10, likeSrtnCd='005930')
    for i in samsung_stock_api:
        print(i["basDt"], i["itmsNm"], i["clpr"], i["trqu"])


try:
    main()
except OpenAPIKeyError as e:
    print("인증키 오류:", e)
except ExceedsError as e:
    print("일일 크롤링 한도 초과 오류:", e)
except PARAMETERERROR as e:
    print("필수 파라미터 미기입 오류:", e)


20260826 삼성전자 261500 19532523
20260825 삼성전자 257000 21617407
20260824 삼성전자 257000 32451940
20260821 삼성전자 281500 27746471
20260820 삼성전자 271000 26095919
20260819 삼성전자 247500 22788552
20260818 삼성전자 268500 24464621
20260814 삼성전자 274500 21669476
20260813 삼성전자 268000 35530867
20260812 삼성전자 255500 27102479


### 문항 2 DB 스키마 설계와 적재 검증

In [43]:
CODES = ["005930", "000660", "035420", "051910", "005380",
         "006400", "035720", "068270", "105560", "055550"]

SOURCE = 'fsc_api'

HASHING = ['basDt', 'srtnCd']

URL = BASE_URL+FUNCTION1

db_config = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', '3306')),
    'user': os.getenv('DB_USER', 'analysis'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME2', 'fsc_db'),
    'charset': 'utf8mb4'
}

def connect():
    conn = pymysql.connect(**db_config)
    return conn

datas = []
for code in CODES:
   data = fetch(page=1, size=300, beginBasDt='20250101', endBasDt='20251231', likeSrtnCd=code)
   datas.extend(data)
   time.sleep(0.7)

df = pd.DataFrame(datas)
collected_at = datetime.now()

rows = []
for item in df.to_dict(orient='records'):
    payload = json.dumps(item, ensure_ascii=False),
    key_str = '|'.join([SOURCE]+[item[c] for c in HASHING])
    content_hash = hashlib.sha256(key_str.encode()).hexdigest()
    rows.append((SOURCE, URL, collected_at, payload, content_hash))

conn = connect()
try:
    cur = conn.cursor()
    INSERT_SQL = """
        INSERT INTO raw_item(source, url, collected_at, payload, content_hash)
            VALUES(%s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                payload = VALUES(payload),
                collected_at = NOW()
    """
    cur.executemany(INSERT_SQL, rows)
    conn.commit()
    cur.close()
    print(f'{len(rows)}건 적재 완료!')

except Exception as e:
    print('알 수 없는 오류 발생:', e)
finally:
    conn.close()


2420건 적재 완료!
